# MedGemma-4B-PT sans fine-tuning — détection 3 classes (RSNA)

**Diagnostic du bug précédent** :
le modèle après fine-tuning est trop lourd lors de l'inférence.

**Correction** :
Puisque MedGemma-4B-PT ne permet pas de produire un fichier JSON pour lui-même, on ne demande plus de JSON au modèle. On lui fait produire une **description libre** de la radiographie (usage prévu du `-pt` : `image + "Findings:"`), puis le code **Python** classe par mots-clés et **construit** le JSON. Résultat : des sorties variées et réellement liées à l'image, et un JSON toujours valide.

**Limite assumée** : un modèle de base non *instruction-tuned* n'est pas cliniquement exact. La classe `uncertain` (« anormal sans opacité ») est difficile à élucider par prompting. C'est un résultat éducatif défendable qui motive le fine-tuning / un modèle `-it` (niveau `COULD` du README).

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 0. Reproductibilité
# ═══════════════════════════════════════════════════════════════════
import os, random

SEED = 42

def set_all_seeds(seed: int = SEED) -> None:
    """Fixe la seed partout (random/numpy/torch/CUDA) pour des runs reproductibles."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_all_seeds(SEED)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(f'Seed fixée à {SEED}')

In [2]:
# ════════════════════════════════════
# STEP 1. Dépendances (une seule fois)
# ════════════════════════════════════
# --- GPU NVIDIA (CUDA) ---
# %pip install -q "transformers>=4.51.3" accelerate bitsandbytes pillow pandas

# --- CPU uniquement (pas de GPU NVIDIA) ---
%pip install -q "transformers>=4.51.3" accelerate pillow pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ═══════════════════════════════════════════════
# STEP 2. Chemins (ancrés sur le notebook) & device
# ═══════════════════════════════════════════════
from pathlib import Path
import torch

MODEL_ID = 'google/medgemma-4b-pt'  # PT imposé

# Racine projet indépendante du cwd. Notebook supposé dans notebooks/.
try:
    NB_DIR = Path(__file__).resolve().parent
except NameError:
    NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent

# Source des cas : CSV (case_id,image_path,label) prioritaire, sinon dossier.
USE_CSV = True
CSV_PATH = PROJECT_ROOT / 'data' / 'RSNA' / 'rsna_train.csv'    # ou train.csv pour évaluer
IMAGES_DIR = PROJECT_ROOT / 'data' / 'RSNA' / 'Training' / 'Images'     # fallback

OUTPUT_DIR = NB_DIR / 'outputs' / 'json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE, USE_4BIT = 'cuda', True
    print(f'GPU : {torch.cuda.get_device_name(0)}')
else:
    DEVICE, USE_4BIT = 'cpu', False
    print('CPU (lent, sans 4-bit).')

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Device : {DEVICE} | 4-bit : {USE_4BIT} | Sortie : {OUTPUT_DIR}')

CPU (lent, sans 4-bit).
PROJECT_ROOT : C:\Users\vicky\Documents\Efrei\ING1-NEW\Projet\Mastercamp\ARVI-RX\repo_git\Solution-Delivery_DS-7B_ARVI-RX
Device : cpu | 4-bit : False | Sortie : C:\Users\vicky\Documents\Efrei\ING1-NEW\Projet\Mastercamp\ARVI-RX\repo_git\Solution-Delivery_DS-7B_ARVI-RX\finetuning\outputs\json


In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 2b. Liste des cas (CSV ou dossier), chemins normalisés
# ═══════════════════════════════════════════════════════════
import pandas as pd

def resolve_image_path(raw: str) -> Path:
    """Normalise un chemin d'image venu du CSV (slashes Windows, relatif au projet)."""
    raw = str(raw).replace('\\', '/')  # backslash Windows -> slash
    p = Path(raw)
    return p if p.is_absolute() else PROJECT_ROOT / p

def load_cases():
    """Charge la liste des cas à traiter : depuis le CSV si dispo, sinon en scannant IMAGES_DIR."""
    if USE_CSV and CSV_PATH.exists():
        df = pd.read_csv(CSV_PATH, dtype=str).fillna('')
        cases = [(r.get('case_id') or Path(r['image_path']).stem,
                  resolve_image_path(r['image_path']),
                  r.get('label') or None) for _, r in df.iterrows()]
        print(f'{len(cases)} cas depuis {CSV_PATH.name}')
        return cases
    pngs = sorted(IMAGES_DIR.glob('*.png')) if IMAGES_DIR.exists() else []
    print(f'{len(pngs)} images depuis {IMAGES_DIR}')
    return [(p.stem, p, None) for p in pngs]

cases = load_cases()
if cases:
    print('Exemple :', cases[0][0], '| existe ?', cases[0][1].exists())

In [5]:
# ══════════════════════════════════════════
# STEP 3. Authentification HuggingFace (gated)
# ══════════════════════════════════════════
from huggingface_hub import login
# login(token='hf_...')   # conditions à accepter sur la model card medgemma-4b-pt
print('En cas de 401/403 : login(token="hf_...").')

En cas de 401/403 : login(token="hf_...").


In [6]:
# ══════════════════════════════════════════════════
# STEP 4. Chargement de MedGemma-4B-PT (inférence)
# ══════════════════════════════════════════════════
import warnings
from transformers import AutoProcessor, AutoModelForImageTextToText
warnings.filterwarnings('ignore')

processor = AutoProcessor.from_pretrained(MODEL_ID)

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map={'': 0}, dtype=torch.bfloat16)
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=torch.float32, low_cpu_mem_usage=True)
    model.to(DEVICE)

model.eval()
print('Modèle chargé.  Device:', next(model.parameters()).device)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Modèle chargé.  Device: cpu


In [7]:
# ═══════════════════════════════════════════════════════════════════
# STEP 5. Prompt de DESCRIPTION LIBRE (fix principal)
# ═══════════════════════════════════════════════════════════════════
# On NE met PAS le schéma JSON ici : sinon le -pt le recopie. On demande
# une description type compte-rendu, ce qu'un modèle pretrained sait
# compléter (usage documenté : image + 'Findings:').

BOI = processor.tokenizer.boi_token  # token image

def build_prompt() -> str:
    return (
        f'{BOI}\n'
        'You are reading a frontal chest X-ray.\n'
        'Describe the lung fields and state whether there is any opacity, '
        'consolidation, or whether the lungs are clear.\n'
        'Findings:'
    )

print('Prompt de description prêt (sans schéma JSON).')
print(build_prompt())

Prompt de description prêt (sans schéma JSON).
<start_of_image>
You are reading a frontal chest X-ray.
Describe the lung fields and state whether there is any opacity, consolidation, or whether the lungs are clear.
Findings:


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 6. Classification par mots-clés + construction du JSON (Python)
# ═══════════════════════════════════════════════════════════════════
# Le JSON n'est PAS généré par le modèle : Python le construit à partir
# de la description libre. Garantit un JSON valide et 3 classes possibles.
import json, re

WARNING_TEXT = ('Educational prototype only. Not for diagnosis. '
                'A qualified clinician must verify the image.')

# (a) Mots-clés complétés. Ajuste-les selon les descriptions réelles observées.
OPACITY_KW = ['opacity', 'opacities', 'opacified', 'opacification',
              'consolidation', 'consolidations', 'pneumonia', 'infiltrate',
              'infiltration', 'airspace', 'air space', 'effusion', 'effusions',
              'haziness', 'hazy', 'nodule', 'nodules', 'mass', 'masses',
              'atelectasis', 'density', 'densities', 'reticular']
NORMAL_KW = ['normal', 'clear', 'unremarkable', 'no acute', 'no focal',
             'within normal limits', 'no abnormal', 'no significant']
# Normalité SPÉCIFIQUE du poumon, pour la règle d'ambiguïté. On n'utilise PAS
# 'normal' ici : ce mot décrit souvent le coeur/les os même en cas d'opacité,
# et casserait les vrais 'suspected_opacity'.
CLEAR_KW = ['clear']
# Langage d'incertitude -> uncertain (seulement en l'absence d'opacité franche).
HEDGE_KW = ['possible', 'possibly', 'may represent', 'cannot exclude',
            'cannot be excluded', 'difficult to assess', 'questionable',
            'equivocal', 'suboptimal', 'underpenetrated', 'poorly',
            'low lung volumes', 'suspicious', 'concerning for']
NEG_TERMS = ['no', 'without', 'free of', 'negative for', 'absence of']

def _positive_hit(text: str, keywords: list[str]) -> str | None:
    """Cherche un mot-clé NON nié (ignore 'no opacity', 'without consolidation')."""
    t = text.lower()
    for k in keywords:
        for m in re.finditer(re.escape(k), t):
            ctx = t[max(0, m.start() - 25):m.start()]
            if any(re.search(rf'\b{re.escape(n)}\b', ctx) for n in NEG_TERMS):
                continue  # occurrence niée -> ne compte pas
            return k
    return None

def classify_from_text(text: str) -> tuple[str, float, list[str]]:
    """Retourne (classe, confiance heuristique, evidence)."""
    op = _positive_hit(text, OPACITY_KW)
    clear = _positive_hit(text, CLEAR_KW)
    hedge = _positive_hit(text, HEDGE_KW)

    # (b) Ambiguïté : opacité ET poumon décrit 'clear' -> description
    # contradictoire (ex. un poumon opacifié, l'autre clair) -> uncertain.
    if op and clear:
        return 'uncertain', 0.50, [f"ambigu : '{op}' + 'clear'"]
    # Opacité franche non contredite -> suspected_opacity.
    if op:
        return 'suspected_opacity', 0.70, [f"terme détecté : '{op}'"]
    # Langage d'incertitude sans opacité -> uncertain.
    if hedge:
        return 'uncertain', 0.50, [f"langage d'incertitude : '{hedge}'"]
    # Normalité affirmée -> normal.
    norm = _positive_hit(text, NORMAL_KW)
    if norm:
        return 'normal', 0.70, [f"terme détecté : '{norm}'"]
    # Aucun signal clair -> garde-fou uncertain (README).
    return 'uncertain', 0.50, ['aucun terme discriminant clair']

def build_record(case_id, image_path, label, cls, conf, evidence, raw_text):
    """Assemble le JSON final d'un cas à partir de la classe déduite et de la description brute."""
    return {
        'case_id': case_id,
        'image_path': str(image_path),
        'label': label,
        'image_quality': 'good',   # heuristique : le -pt n'évalue pas la qualité de façon fiable
        'predicted_class': cls,
        'confidence': round(float(conf), 3),
        'visual_evidence': evidence + ([raw_text.strip()[:200]] if raw_text.strip() else []),
        'justification': ('Classe déduite par mots-clés depuis la description libre générée '
                          'par MedGemma-4b-pt. Résultat expérimental, non clinique.'),
        'limitations': [
            'modèle de base (pt) non instruction-tuned',
            'classification par mots-clés, non calibrée',
            'confiance heuristique, non probabiliste',
            'pas de contexte clinique',
        ],
        'warning': WARNING_TEXT,
    }

print('Classifieur (mots-clés complétés + règle d\'ambiguïté) + constructeur JSON prêts.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 7. Inférence sur une image : description -> classe -> JSON
# ═══════════════════════════════════════════════════════════
from PIL import Image

def predict(case_id, image_path, label=None, debug=False) -> dict:
    """Fait générer une description de la radio par MedGemma, puis la reclasse par mots-clés."""
    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, text=build_prompt(),
                       return_tensors='pt').to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, do_sample=False)

    gen = out[0][inputs['input_ids'].shape[1]:]  # ne garder que les tokens générés, pas le prompt
    raw_text = processor.decode(gen, skip_special_tokens=True)

    if debug:
        print('--- DESCRIPTION BRUTE GÉNÉRÉE ---')
        print(raw_text[:500])
        print('--- FIN ---')

    cls, conf, evidence = classify_from_text(raw_text)
    return build_record(case_id, image_path, label, cls, conf, evidence, raw_text)

print('predict() prête.')

In [10]:
# ═══════════════════════════════════════════════════════════════════
# STEP 8. Test sur 1 image — AVEC debug (voir la description réelle)
# ═══════════════════════════════════════════════════════════════════
if cases:
    cid, path, lbl = cases[0]
    print(f'Test : {cid}  (label connu : {lbl})\n')
    rec = predict(cid, path, lbl, debug=True)
    print()
    print(json.dumps(rec, indent=2, ensure_ascii=False))
else:
    print('Aucun cas — vérifie CSV_PATH / IMAGES_DIR (STEP 2).')

[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.


Test : 0004cfab-14fd-4e49-80ba-63a80b6bddd6  (label connu : None)

--- DESCRIPTION BRUTE GÉNÉRÉE ---

There is a right-sided chest tube with a small apical pneumothorax.
There is a right-sided pleural effusion.
There is a right lower lobe consolidation.
There is a left lower lobe consolidation.
There is a nodular opacity in the left upper lobe.
There is a nodular opacity in the right upper lobe.
There is a nodular opacity in the right lower lobe.
There is a nodular opacity in the left lower lobe.
There is a nodular opacity in the right upper lobe.
There is a nodular opacity in the left
--- FIN ---

{
  "case_id": "0004cfab-14fd-4e49-80ba-63a80b6bddd6",
  "image_path": "C:\\Users\\vicky\\Documents\\Efrei\\ING1-NEW\\Projet\\Mastercamp\\ARVI-RX\\repo_git\\Solution-Delivery_DS-7B_ARVI-RX\\data\\RSNA\\Training\\Images\\0004cfab-14fd-4e49-80ba-63a80b6bddd6.png",
  "label": null,
  "image_quality": "good",
  "predicted_class": "suspected_opacity",
  "confidence": 0.7,
  "visual_evidence": [
 

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 9. Inférence en lot -> 1 JSON/image + distribution
# ═══════════════════════════════════════════════════════════
from collections import Counter

def run_batch(cases, out_dir=OUTPUT_DIR, limit=None):
    """Lance predict() sur une liste de cas, écrit un JSON par image et affiche la distribution des classes.

    Une exception sur un cas isolé (image corrompue, timeout...) ne fait pas
    planter tout le lot : le cas est simplement enregistré en "uncertain" avec
    l'exception en guise de justification, et le batch continue.
    """
    items = cases[:limit] if limit else cases
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    dist, written = Counter(), []
    for i, (cid, path, lbl) in enumerate(items, 1):
        try:
            rec = predict(cid, path, lbl)
        except Exception as e:
            rec = build_record(cid, path, lbl, 'uncertain', 0.0,
                               [f'exception: {e}'], '')
        with open(out_dir / f'{cid}.json', 'w', encoding='utf-8') as f:
            json.dump(rec, f, indent=2, ensure_ascii=False)
        dist[rec['predicted_class']] += 1
        written.append(cid)
        print(f"[{i}/{len(items)}] {cid} -> {rec['predicted_class']} "
              f"(conf {rec['confidence']})")
    print(f'\n{len(written)} JSON écrits dans {out_dir}')
    print('Distribution des classes :', dict(dist))
    return written

# 10 premiers (retire limit pour tout traiter). CPU : compte ~plusieurs s/image.
files = run_batch(cases, limit=10)

## Lire et régler les résultats

La **distribution des classes** (STEP 9) doit maintenant montrer plusieurs classes, pas uniquement `uncertain`. Chaque JSON contient la **description réelle** générée (`visual_evidence`), donc traçable.

**Si tout reste `uncertain`** → regarde la description brute du STEP 8. Deux cas :

- La description est pertinente mais utilise d'autres mots → **ajoute-les** aux listes `OPACITY_KW` / `NORMAL_KW` (STEP 6). C'est le réglage principal.
- La description est vide ou hors-sujet → augmente `max_new_tokens` (STEP 7) ou reformule le prompt (STEP 5).

**Sur la classe `uncertain` (RSNA « No Lung Opacity / Not Normal »)** : elle est intrinsèquement ambiguë et un modèle base ne la distingue quasiment pas de `normal` par simple description. Un taux d'erreur élevé sur cette classe est attendu — à documenter comme limite, pas à masquer (cf. README : « ne pas afficher uniquement des réussites »).

**Confiance** : elle est **heuristique** (0.70 si mot-clé franc, 0.50 sinon), pas une probabilité calibrée. Un modèle `-pt` ne fournit pas de confiance fiable ; c'est une limite honnête du prototype sans fine-tuning.